In [4]:
import boto3
import json

In [7]:
bedrock = boto3.client(service_name='bedrock-runtime')
bedrock_agent = boto3.client(service_name='bedrock-agent')

MODEL_ID = "amazon.nova-micro-v1:0"

In [8]:
temperature = .7

inference_config = {"temperature": temperature}

system_prompts = [{"text": "You are a virtual travel assistant that suggests destinations based on user preferences."
                + "Only return destination names and a brief description."}]

messages = []

message_1 = {
    "role": "user",
    "content": [{"text": "Create a list of 3 travel destinations."}]
}

messages.append(message_1)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

def print_response(response):
    model_response = response.get('output', {}).get('message', {}).get('content', [{}])[0].get('text', '')

    print("✈️ Your suggested travel destinations:")
    print(model_response)

print_response(response)

✈️ Your suggested travel destinations:
1. **Kyoto, Japan**
   - Experience traditional Japanese culture with stunning temples, serene gardens, and historic tea houses.

2. **Banff National Park, Canada**
   - Enjoy breathtaking mountain scenery, outdoor activities like hiking and skiing, and the charming town of Banff.

3. **Santorini, Greece**
   - Discover picturesque white-washed buildings, stunning sunsets, and crystal-clear waters on this iconic island.


In [9]:
message_2 = {
        "role": "user",
        "content": [{"text": "Only suggest travel locations that are no more than one short flight away."}]
}

messages.append(message_2)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Orlando, Florida**  
   Experience world-famous theme parks like Disney World and Universal Studios.

2. **Miami, Florida**  
   Enjoy vibrant beaches, rich culture, and lively nightlife.

3. **San Diego, California**  
   Discover beautiful beaches, the San Diego Zoo, and stunning coastal views.


In [12]:
message_3 = {
        "role": "user",
        "content": [{"text": "List destinations by their proximity to Augusta, GA USA. Recall Augusta, GA only has direct flights to Atlanta, GA, Charlotte, NC, and Dallas, TX, and Washington D.C. Driving is always an option."}]
}

messages.append(message_3)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Atlanta, GA** - Just a short flight away, Atlanta offers a vibrant mix of Southern charm, rich history, and a bustling cultural scene with attractions like the World of Coca-Cola and the Georgia Aquarium.

2. **Charlotte, NC** - Another short flight option, Charlotte is known for its thriving arts scene, beautiful parks, and the iconic Carolina Panthers football team.

3. **Washington D.C.** - A bit further but still reachable by a short flight, Washington D.C. is a hub for political history, world-class museums, and iconic landmarks like the Lincoln Memorial and the U.S. Capitol.


In [13]:
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Checks if all trip information has been provided.",
        variants=[
            { 
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": { 
                    "chat": {
                        'system': [ 
                            {
                                "text": """You are a travel agent evaluating trip requests for custom itineraries. 
                                Review the message carefully and answer YES or NO to the following screening questions. 
                                Be strict—if any detail is missing or unclear, answer NO.

                                A) Is the destination clearly stated?
                                B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
                                C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
                                D) Is there any mention of a valid passport or travel documentation?
                                E) Is there enough information to follow up with a proposed itinerary?"""
                            }
                        ],
                        'messages': [{
                            'role': 'user',
                            'content': [ 
                                {
                                    'text': "Trip request: {{event_request}}"
                                }
                            ]
                        }],
                        'inputVariables' : [
                            { 'name' : 'event_request'}
                        ]
                    }
                }
        }]
    )
    print("Created!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TripBooker_xyz"), None)
    prompt_arn = prompt['arn']

prompt_arn

Created!


'arn:aws:bedrock:us-east-1:458806987020:prompt/DZJDLVWJ5H'

In [14]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={
        'event_request': {
            'text': """
                Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days in Florence and Venice as well. We’d love recommendations on tours, cultural sites, and good local restaurants. We’re not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!
                """
        }
    },
)
print(response['output']['message']['content'][0]['text'])

YES

A) Is the destination clearly stated?
YES. The destination is clearly stated as Italy.

B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
YES. The travel dates are within a reasonable range, September 10–20 this year.

C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
YES. The request specifies a preference for a relaxing and enriching experience without any risky activities like skydiving or remote hiking.

D) Is there any mention of a valid passport or travel documentation?
YES. Both travelers have valid passports.

E) Is there enough information to follow up with a proposed itinerary?
YES. There is enough information provided to create a proposed itinerary, including the destination, travel dates, preferred cities, and general interests in tours, cultural sites, and local restaurants.
